# Estimate noise correlations along the x, y, t, T axes before and after pre-denosing

## Define mask

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath("../src"))
from denoising.eval.empirical_noise_correlations import *
from denoising.figures.empirical_noise_correlations_plots import *
from denoising.data.data_utils import *


# Configuration cell

In [ ]:
# this is the only cell where you need to adapt parameters

#subject_ids = [f"P0{i}" for i in range(3, 4)]

#subject_ids = [f"Simulated_Lesion_double_{i}" for i in range(1, 2)]

subject_ids = ["Vol5/OriginalData"]

methods = {
    "No denoising": {"type": "raw"},
    #"tMPPCA": {"type": "precomputed", "method": "tMPPCA_5D"},
    #"Spin SVD": {"type": "callable", "fn": apply_low_rank_5d_to_dataset_list, "kwargs": {"rank": 8}},
}

#configure spatial noise mask:
mask_source = "method"  # "raw" or "method": raw from the noisy data, method from the method directly
mask_type = "spatial"#"fid_window" # fid_window, fid or spatial possible
fid_window = (0.55, 0.75)

# no effect for mask type fid_window
percentile = 5

## datasets
suffix = "" #"normalized" #"normalized_uncorrelated_noise"
npy_name = "data.npy"   #"data.npy"   "data_after_walinet.npy"
base_dir = "../datasets/Proton/B0corrected_wo_LipidMask" #"../datasets  fn_251009_WB_DMI_P01_4pos"

# correlation settings
max_lag = 5 # only applies to 2D correlations
compute_pairwise = True
return_pair_counts = True
compute_spatial = True

In [ ]:
results = run_noise_analysis_pipeline(
    subject_ids=subject_ids,
    methods=methods,
    mask_source=mask_source,
    mask_type=mask_type,
    fid_window=fid_window,
    percentile=percentile,
    suffix=suffix,
    base_dir=base_dir,
    npy_name = npy_name,
    max_lag=max_lag,
    compute_spatial=compute_spatial,
    compute_pairwise=compute_pairwise,
    return_pair_counts=return_pair_counts,
    subtract_mean=True,
    normalize=True,
)

In [ ]:
# check that noise mask makes sense: (only applicaple to fid_window)

plot_mean_fid_with_noise_mask(
    results,
    methods=["No denoising"],
    title="Mean FID and selected noise region"
)

# First: 1D Autocorrelation along each of the 5 axes

We estimate the 1D autocorrelation of the data along each axis $a \in \{x, y, z, t, T\}$.

For a given axis $a$ and lag $\ell$, the autocorrelation is defined as

$$
R_a(\ell)
=
\mathbb{E}\left[
\bigl(X(i) - \mu\bigr)\,
\overline{\bigl(X(i+\ell_a) - \mu\bigr)}
\right],
$$

where:
- $X$ denotes the data array,
- $i$ is a multi-dimensional index,
- $i+\ell_a$ denotes a shift by $\ell$ along axis $a$,
- $\mu$ is the mean of the noise (often approximately zero),
- $\overline{(\cdot)}$ denotes complex conjugation.

In practice, the expectation is approximated by averaging over all valid index pairs:

$$
R_a(\ell)
\approx
\frac{1}{N_\ell}
\sum_{i \in \mathcal{I}_\ell}
\bigl(X(i) - \mu\bigr)\,
\overline{\bigl(X(i+\ell_a) - \mu\bigr)},
$$

where:
- $\mathcal{I}_\ell$ is the set of valid index pairs (both inside the domain and inside the noise mask),
- $N_\ell = |\mathcal{I}_\ell|$ is the number of valid pairs.

Finally, the autocorrelation is typically normalized such that

$$
R_a(0) = 1.
$$

This measures how strongly the noise is correlated as a function of distance (lag) along each axis.

In [ ]:
plot_spatial_acf_comparison(
    results["spatial_stats_by_method"],
    axes_to_plot=(0, 1, 2),
    max_lag_plot=15,
    title="Spatial noise correlations"
)

In [ ]:
plot_acf_comparison(
    results["acf_stats_by_method"],
    axis_indices=[i for i in results["axis_indices"] if i not in [0,1,2]],
    axis_names=results["axis_names"],
    title="Noise autocorrelation"
)

# Second: 2D (pairwise) autocorrelation between axes

To analyze correlations jointly across two axes, we estimate the 2D autocorrelation function for pairs of axes $(a, b)$.

For given lags $(\ell_a, \ell_b)$ along axes $a$ and $b$, the joint autocorrelation is defined as

$$
R_{a,b}(\ell_a, \ell_b)
=
\mathbb{E}\left[
\bigl(X(i) - \mu\bigr)\,
\overline{\bigl(X(i+\ell_a+\ell_b) - \mu\bigr)}
\right],
$$

where:
- $X$ denotes the data array,
- $i$ is a multi-dimensional index,
- $i+\ell_a+\ell_b$ denotes a shift by $\ell_a$ along axis $a$ and by $\ell_b$ along axis $b$,
- $\mu$ is the mean of the noise,
- $\overline{(\cdot)}$ denotes complex conjugation.

In practice, the expectation is approximated by averaging over all valid index pairs:

$$
R_{a,b}(\ell_a, \ell_b)
\approx
\frac{1}{N_{\ell_a,\ell_b}}
\sum_{i \in \mathcal{I}_{\ell_a,\ell_b}}
\bigl(X(i) - \mu\bigr)\,
\overline{\bigl(X(i+\ell_a+\ell_b) - \mu\bigr)},
$$

where:
- $\mathcal{I}_{\ell_a,\ell_b}$ is the set of valid index pairs (both samples inside the domain and inside the noise mask),
- $N_{\ell_a,\ell_b} = |\mathcal{I}_{\ell_a,\ell_b}|$ is the number of valid pairs.

The result is a 2D function over lag pairs $(\ell_a, \ell_b)$, describing how the noise is correlated under simultaneous shifts along both axes.

As in the 1D case, the autocorrelation is typically normalized such that

$$
R_{a,b}(0,0) = 1.
$$

This allows comparison of correlation structure across different axis pairs and datasets.

In [ ]:
for method in results["pair_corr_stats_by_method"].keys():
    plot_pairwise_correlations(results, method=method, panel_size=2.5)